# 0. Imports & Config Read

In [21]:
# External Dependencies
import pandas as pd
from pprint import pprint

# Internal Imports
from src.data_collectors.collector_yfin import YFinanceNSEPipeline
from src.modelling.var_engine import LeadLagVAREngine
from src.utils.file_operators import load_yaml

C:\Users\sharv\Documents\Sharvil\Projects\lead-lag-vector-auto-regression\src\modelling\var_engine.py:593: SyntaxWarning: invalid escape sequence '\h'
  """


In [22]:
# Load the catalog and params from yaml to dict
config_catalog = load_yaml("../config_catalog.yml")
config_params = load_yaml("../config_parameters.yml")

## 0.1. Parameters

In [23]:
pprint(config_params)

{'adf_testing_params': {'alpha': 0.05},
 'market_params': {'end_time': '15:30',
                   'start_time': '09:30',
                   'time_zone': 'Asia/Kolkata'},
 'time_params': {'interval': '1m', 'period': '7d'},
 'universe_params': {'tech_eqs': ['TCS', 'INFY', 'MPHASIS', 'LTIM', 'COFORGE'],
                     'universe_name': 'tech_eqs'},
 'var_params': {'alpha': 0.05,
                'estimation_method': 'ols',
                'ic_criterion': 'aic',
                'max_lags': 10}}


## 0.2. Catalog

In [24]:
pprint(config_catalog)

{'data': {'collector_yfin_cleaned_data': {'directory': 'C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regression/data/collector_yfin/01_processed',
                                          'file_format': 'csv',
                                          'file_name': 'cleaned_data',
                                          'versioned': True},
          'collector_yfin_log_returns': {'directory': 'C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regression/data/collector_yfin/01_processed',
                                         'file_format': 'csv',
                                         'file_name': 'log_returns_data',
                                         'versioned': True},
          'collector_yfin_raw_data': {'directory': 'C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regression/data/collector_yfin/00_raw_data',
                                      'file_format': 'csv',
                                      'file_name': 'raw_

# 1. Data Load and QC

In [25]:
# Initialise a data loader obj
data_loader = YFinanceNSEPipeline(
    config_catalog=config_catalog,
    config_params=config_params
)

In [26]:
# Fetch Data
data_loader.fetch_raw_data()

INFO: [INGESTION]: Fetching 7d of 1m data for 5 assets from Yahoo Finance (Sequential Mode)
INFO: [INGESTION]: Downloading TCS (TCS.NS)...
SUCCESS:   └─ Received 2526 bars for TCS
INFO: [INGESTION]: Downloading INFY (INFY.NS)...
SUCCESS:   └─ Received 2526 bars for INFY
INFO: [INGESTION]: Downloading MPHASIS (MPHASIS.NS)...
SUCCESS:   └─ Received 2486 bars for MPHASIS
INFO: [INGESTION]: Downloading LTIM (LTIM.NS)...


$LTIM.NS: No data found, symbol may be delisted


WARN:   └─ Warning: Empty or invalid payload for LTIM
INFO: [INGESTION]: Downloading COFORGE (COFORGE.NS)...
SUCCESS:   └─ Received 2525 bars for COFORGE
SUCCESS: [INGESTION]: Successfully downloaded 2527 raw rows across 4 valid assets from Yahoo Finance


,TCS,INFY,MPHASIS,COFORGE
Datetime,,,,
2026-08-17 09:15:00+05:30,2354.800049,1162.599976,2542.600098,1818.699951
2026-08-17 09:16:00+05:30,2348.399902,1162.199951,2532.899902,1814.400024
2026-08-17 09:17:00+05:30,2344.899902,1162.500000,2525.800049,1812.599976
2026-08-17 09:18:00+05:30,2340.600098,1161.000000,2525.000000,1809.099976
2026-08-17 09:19:00+05:30,2340.899902,1160.800049,2528.300049,1808.000000
...,...,...,...,...
2026-08-25 15:11:00+05:30,2287.500000,1130.599976,2408.199951,1899.000000
2026-08-25 15:12:00+05:30,2287.899902,1130.800049,2405.100098,1898.599976
2026-08-25 15:13:00+05:30,2288.000000,1131.000000,2407.000000,1898.000000


In [27]:
# Align and Process Data
data_loader.align_and_clean_data()

INFO: [PROCESSING]: Processing 7d of 1m data for 5 assets
SUCCESS: [PROCESSING] Aligned grid size: 2422 rows across 5 stocks.


,TCS,INFY,MPHASIS,COFORGE
Datetime,,,,
2026-08-17 09:30:00+05:30,2325.699951,1152.300049,2502.600098,1804.000000
2026-08-17 09:31:00+05:30,2322.800049,1151.000000,2500.800049,1804.500000
2026-08-17 09:32:00+05:30,2321.899902,1150.900024,2501.500000,1806.199951
2026-08-17 09:33:00+05:30,2324.899902,1151.199951,2502.000000,1805.000000
2026-08-17 09:34:00+05:30,2327.899902,1152.199951,2503.800049,1805.900024
...,...,...,...,...
2026-08-25 15:11:00+05:30,2287.500000,1130.599976,2408.199951,1899.000000
2026-08-25 15:12:00+05:30,2287.899902,1130.800049,2405.100098,1898.599976
2026-08-25 15:13:00+05:30,2288.000000,1131.000000,2407.000000,1898.000000


In [28]:
# Compute Log Returns
data_loader.compute_log_returns()

INFO: [PROCESSING]: Computing Log Returns for 5 assets


,TCS,INFY,MPHASIS,COFORGE
Datetime,,,,
2026-08-17 09:31:00+05:30,-0.001248,-0.001129,-0.000720,0.000277
2026-08-17 09:32:00+05:30,-0.000388,-0.000087,0.000280,0.000942
2026-08-17 09:33:00+05:30,0.001291,0.000261,0.000200,-0.000665
2026-08-17 09:34:00+05:30,0.001290,0.000868,0.000719,0.000499
2026-08-17 09:35:00+05:30,-0.000473,-0.000521,0.000240,0.001107
...,...,...,...,...
2026-08-25 15:11:00+05:30,-0.000087,0.000088,-0.000747,0.000053
2026-08-25 15:12:00+05:30,0.000175,0.000177,-0.001288,-0.000211
2026-08-25 15:13:00+05:30,0.000044,0.000177,0.000790,-0.000316


In [29]:
# Perform ADF test
data_loader.validate_stationarity()

INFO: [ANALYSING]: Performing ADF for 5 assets


,ADF Statistic,p-value,Stationary (I(0))
TCS,-50.2157,0.0,True
INFY,-50.3566,0.0,True
MPHASIS,-52.4037,0.0,True
COFORGE,-17.1458,0.0,True


# 2. Fit and Model VAR

In [30]:
# Initialise a data loader obj
var_engine = LeadLagVAREngine(
    config_catalog=config_catalog,
    config_params=config_params
)

In [31]:
# Fit the VAR
var_engine.fit_var_model()

SUCCESS: [VAR ENGINE]: Selected optimal lag p = 2 minute(s) via 'AIC' criterion.
INFO: [VAR Engine]: Fitted VAR(2) using OLS.


{'optimal_lag': np.int64(2),
 'method': 'ols',
 'phi_matrice_path': WindowsPath('C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regression/data/var_engine/01_weights/phi_matrices_tech_eqs_20260825_233542.csv'),
 'phi_matrie':                  TCS          INFY   MPHASIS   COFORGE
 const      -0.000002  6.298306e-07 -0.000012  0.000025
 L1.TCS     -0.092693  1.490360e-01  0.049875  0.009415
 L1.INFY     0.018140 -1.609454e-01 -0.022136  0.015606
 L1.MPHASIS  0.044823  4.112670e-02 -0.097745  0.007007
 L1.COFORGE  0.041018  6.974356e-02  0.073755 -0.047193
 L2.TCS     -0.079689 -9.175705e-02 -0.060568 -0.105117
 L2.INFY     0.043731  5.443582e-02  0.075833  0.031895
 L2.MPHASIS  0.100199  1.429466e-01  0.019013  0.177419
 L2.COFORGE -0.079237 -1.019713e-01 -0.064860 -0.090523,
 'fitted_model': <statsmodels.tsa.vector_ar.var_model.VARResultsWrapper at 0x2249fdcc4a0>,
 'fitted_model_path': WindowsPath('C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regressio

In [32]:
# Compute to checck grangers causality
var_engine.compute_grangers_causality()

INFO: [VAR ENGINE]: Computing pairwise Granger Causality tests across 4 assets...
SUCCESS: [VAR ENGINE]: Successfully computed Granger Causality p-value matrix.

       GRANGER CAUSALITY QUANTITATIVE INTERPRETATION (Alpha = 0.05)

[1] PAIRWISE DIRECTIONAL BREAKDOWN:

[VALID LINK]    TCS vs INFY:
    - p(TCS -> INFY): 8e-06
    - p(INFY -> TCS): 0.101557
    - Classification : STRICT LEAD-LAG
    - Action         : Trade INFY using TCS forecast

[VALID LINK]    TCS vs MPHASIS:
    - p(TCS -> MPHASIS): 0.058142
    - p(MPHASIS -> TCS): 1e-06
    - Classification : STRICT LEAD-LAG (REVERSE)
    - Action         : Trade TCS using MPHASIS forecast

[FEEDBACK LOOP] TCS vs COFORGE:
    - p(TCS -> COFORGE): 0.024984
    - p(COFORGE -> TCS): 1e-06
    - Classification : FEEDBACK LOOP
    - Action         : BLOCK / REDUCE WEIGHT (Systemic Co-movement)

[FEEDBACK LOOP] INFY vs MPHASIS:
    - p(INFY -> MPHASIS): 0.006958
    - p(MPHASIS -> INFY): 2e-06
    - Classification : FEEDBACK LOOP
    - Ac

,TCS,INFY,MPHASIS,COFORGE
TCS,1.000000,0.000008,0.058142,0.024984
INFY,0.101557,1.000000,0.006958,0.526537
MPHASIS,0.000001,0.000002,1.000000,0.000000
COFORGE,0.000001,0.000000,0.000014,1.000000


In [ ]:
# Forecast using base VAR model
var_engine.generate_next_bar_signal()

SUCCESS: [VAR ENGINE]: Successfully generated 1-step-ahead return forecasts for 'tech_eqs'.


,TCS,INFY,MPHASIS,COFORGE
forecast_return_t_plus_1,0.00114,0.000478,0.000305,0.002294
